# Environment Compatibility Check 🔍

This notebook verifies that your MLU environment is properly set up and compatible with your existing Anaconda and Docker installations.

## 📋 What This Notebook Does

1. **System Information**: Checks OS, Python version, and hardware
2. **Package Verification**: Ensures all required packages are installed
3. **GPU Detection**: Verifies CUDA availability and configuration
4. **Environment Details**: Shows conda/Docker environment information
5. **Performance Baseline**: Runs basic tensor operations for benchmarking

**Run this notebook first to ensure your environment is ready for deep learning!**

## 1. System Information and Environment Detection 💻

In [ ]:
import sys
import os
import platform
import subprocess
import json
from datetime import datetime

# Check if we're in various environments
def detect_environment():
    """Detect the current execution environment"""
    env_info = {
        'python_version': sys.version,
        'python_executable': sys.executable,
        'platform': platform.platform(),
        'architecture': platform.architecture(),
        'processor': platform.processor(),
        'os': platform.system(),
        'release': platform.release(),
        'timestamp': datetime.now().isoformat()
    }
    
    # Check for Jupyter environments
    try:
        import IPython
        env_info['jupyter'] = True
        env_info['ipython_version'] = IPython.__version__
    except ImportError:
        env_info['jupyter'] = False
    
    # Check for Google Colab
    try:
        import google.colab
        env_info['colab'] = True
    except ImportError:
        env_info['colab'] = False
    
    # Check for Docker
    if os.path.exists('/.dockerenv'):
        env_info['docker'] = True
    else:
        env_info['docker'] = False
    
    # Check for conda
    env_info['conda'] = os.environ.get('CONDA_DEFAULT_ENV') is not None
    if env_info['conda']:
        env_info['conda_env'] = os.environ.get('CONDA_DEFAULT_ENV')
    
    # Check for virtual environment
    env_info['venv'] = hasattr(sys, 'real_prefix') or (
        hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
    )
    
    return env_info

# Get system information
env_details = detect_environment()

print("🖥️  SYSTEM INFORMATION")
print("=" * 50)
print(f"Operating System: {env_details['os']} {env_details['release']}")
print(f"Platform: {env_details['platform']}")
print(f"Architecture: {env_details['architecture'][0]}")
print(f"Processor: {env_details['processor']}")
print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Python Executable: {env_details['python_executable']}")
print()

print("🏃 EXECUTION ENVIRONMENT")
print("=" * 50)
print(f"Jupyter Notebook: {'✅' if env_details['jupyter'] else '❌'}")
print(f"Google Colab: {'✅' if env_details['colab'] else '❌'}")
print(f"Docker Container: {'✅' if env_details['docker'] else '❌'}")
print(f"Conda Environment: {'✅' if env_details['conda'] else '❌'}")
if env_details['conda']:
    print(f"Conda Environment Name: {env_details.get('conda_env', 'Unknown')}")
print(f"Virtual Environment: {'✅' if env_details['venv'] else '❌'}")
print()

# Save environment info for reference
with open('environment_check_results.json', 'w') as f:
    json.dump(env_details, f, indent=2)

print("💾 Environment details saved to: environment_check_results.json")

## 2. Package Verification and Installation Check 📦

In [ ]:
import importlib
import pkg_resources

def check_package(package_name, import_name=None, required_version=None):
    """Check if a package is installed and optionally verify version"""
    if import_name is None:
        import_name = package_name
    
    try:
        # Try to import the package
        module = importlib.import_module(import_name)
        
        # Get version if available
        try:
            if hasattr(module, '__version__'):
                version = module.__version__
            else:
                # Try to get version from pkg_resources
                version = pkg_resources.get_distribution(package_name).version
        except:
            version = "Unknown"
        
        # Check version requirement if specified
        version_ok = True
        if required_version and version != "Unknown":
            try:
                from packaging import version as pkg_version
                version_ok = pkg_version.parse(version) >= pkg_version.parse(required_version)
            except:
                version_ok = True  # Can't check, assume OK
        
        return {
            'installed': True,
            'version': version,
            'version_ok': version_ok,
            'module': module
        }
    except ImportError:
        return {
            'installed': False,
            'version': None,
            'version_ok': False,
            'module': None
        }

# Required packages for MLU
required_packages = [
    ('torch', 'torch', '1.9.0'),
    ('torchvision', 'torchvision', '0.10.0'),
    ('d2l', 'd2l', '0.17.0'),
    ('numpy', 'numpy', '1.21.0'),
    ('pandas', 'pandas', '1.3.0'),
    ('matplotlib', 'matplotlib', '3.5.0'),
    ('seaborn', 'seaborn', '0.11.0'),
    ('scikit-learn', 'sklearn', '1.0.0'),
    ('jupyter', 'jupyter', None),
    ('jupyterlab', 'jupyterlab', None),
    ('ipywidgets', 'ipywidgets', None),
    ('tqdm', 'tqdm', None),
    ('plotly', 'plotly', None),
]

print("📦 PACKAGE VERIFICATION")
print("=" * 80)
print(f"{'Package':<15} {'Status':<10} {'Version':<15} {'Min Required':<15} {'Notes':<20}")
print("-" * 80)

package_status = {}
all_good = True

for package_name, import_name, min_version in required_packages:
    result = check_package(package_name, import_name, min_version)
    package_status[package_name] = result
    
    status = "✅ OK" if result['installed'] else "❌ Missing"
    version = result['version'] if result['version'] else "N/A"
    min_req = min_version if min_version else "Any"
    
    notes = ""
    if result['installed']:
        if min_version and not result['version_ok']:
            notes = "⚠️ Old version"
            all_good = False
    else:
        notes = "❌ Install needed"
        all_good = False
    
    print(f"{package_name:<15} {status:<10} {version:<15} {min_req:<15} {notes:<20}")

print("-" * 80)
if all_good:
    print("🎉 All packages are properly installed!")
else:
    print("⚠️ Some packages need attention. See notes above.")
print()

# Special checks for GPU packages
print("🎮 GPU PACKAGE VERIFICATION")
print("=" * 50)

# Check CUDA availability in PyTorch
if package_status['torch']['installed']:
    torch = package_status['torch']['module']
    cuda_available = torch.cuda.is_available()
    print(f"PyTorch CUDA Available: {'✅' if cuda_available else '❌'}")
    
    if cuda_available:
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"GPU Count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    else:
        print("ℹ️ CPU-only PyTorch detected")
else:
    print("❌ PyTorch not installed - cannot check GPU support")

print()

# Installation suggestions if needed
if not all_good:
    print("🔧 INSTALLATION SUGGESTIONS")
    print("=" * 50)
    
    missing_packages = [name for name, status in package_status.items() if not status['installed']]
    if missing_packages:
        print("Missing packages can be installed with:")
        print()
        
        if env_details['conda']:
            print("Using conda:")
            conda_packages = [p for p in missing_packages if p in ['torch', 'torchvision', 'numpy', 'pandas', 'matplotlib', 'scikit-learn', 'jupyter']]
            pip_packages = [p for p in missing_packages if p not in conda_packages]
            
            if conda_packages:
                print(f"conda install {' '.join(conda_packages)}")
            if pip_packages:
                print(f"pip install {' '.join(pip_packages)}")
        else:
            print("Using pip:")
            print(f"pip install {' '.join(missing_packages)}")
    
    print()

print("✅ Package verification complete!")

## 3. GPU and Hardware Verification 🎮

In [ ]:
import psutil
import subprocess
import time

def get_gpu_info():
    """Get detailed GPU information"""
    gpu_info = []
    
    try:
        # Try nvidia-smi first
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used,temperature.gpu,utilization.gpu', 
                               '--format=csv,noheader,nounits'], 
                               capture_output=True, text=True, timeout=10)
        
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            for i, line in enumerate(lines):
                if line.strip():
                    parts = line.split(', ')
                    if len(parts) >= 5:
                        gpu_info.append({
                            'id': i,
                            'name': parts[0].strip(),
                            'memory_total': f"{parts[1].strip()} MB",
                            'memory_used': f"{parts[2].strip()} MB",
                            'temperature': f"{parts[3].strip()}°C",
                            'utilization': f"{parts[4].strip()}%"
                        })
    except (subprocess.TimeoutExpired, FileNotFoundError, subprocess.SubprocessError):
        pass
    
    return gpu_info

def get_system_specs():
    """Get system specifications"""
    specs = {}
    
    # CPU info
    specs['cpu_count'] = psutil.cpu_count(logical=True)
    specs['cpu_count_physical'] = psutil.cpu_count(logical=False)
    specs['cpu_freq'] = psutil.cpu_freq()
    
    # Memory info
    memory = psutil.virtual_memory()
    specs['memory_total'] = f"{memory.total / (1024**3):.1f} GB"
    specs['memory_available'] = f"{memory.available / (1024**3):.1f} GB"
    specs['memory_percent_used'] = f"{memory.percent:.1f}%"
    
    # Disk info
    disk = psutil.disk_usage('/')
    specs['disk_total'] = f"{disk.total / (1024**3):.1f} GB"
    specs['disk_free'] = f"{disk.free / (1024**3):.1f} GB"
    specs['disk_percent_used'] = f"{(disk.used / disk.total) * 100:.1f}%"
    
    return specs

print("🖥️ SYSTEM SPECIFICATIONS")
print("=" * 80)

# Get system specs
system_specs = get_system_specs()

print(f"CPU Cores (Logical): {system_specs['cpu_count']}")
print(f"CPU Cores (Physical): {system_specs['cpu_count_physical']}")
if system_specs['cpu_freq']:
    print(f"CPU Frequency: {system_specs['cpu_freq'].current:.1f} MHz (max: {system_specs['cpu_freq'].max:.1f} MHz)")

print(f"\nMemory Total: {system_specs['memory_total']}")
print(f"Memory Available: {system_specs['memory_available']}")
print(f"Memory Used: {system_specs['memory_percent_used']}")

print(f"\nDisk Total: {system_specs['disk_total']}")
print(f"Disk Free: {system_specs['disk_free']}")
print(f"Disk Used: {system_specs['disk_percent_used']}")

print("\n🎮 GPU INFORMATION")
print("=" * 80)

# Get GPU info
gpu_info = get_gpu_info()

if gpu_info:
    for gpu in gpu_info:
        print(f"GPU {gpu['id']}: {gpu['name']}")
        print(f"  Memory: {gpu['memory_used']} / {gpu['memory_total']}")
        print(f"  Temperature: {gpu['temperature']}")
        print(f"  Utilization: {gpu['utilization']}")
        print()
else:
    print("No NVIDIA GPU detected or nvidia-smi not available")
    print("ℹ️ This doesn't prevent deep learning, CPU training is possible")

# Performance recommendations
print("\n🚀 PERFORMANCE RECOMMENDATIONS")
print("=" * 80)

# Memory check
total_memory_gb = float(system_specs['memory_total'].split()[0])
if total_memory_gb >= 16:
    print("✅ Memory: Excellent (16GB+) - Can handle large models and datasets")
elif total_memory_gb >= 8:
    print("✅ Memory: Good (8-16GB) - Suitable for most learning tasks")
else:
    print("⚠️ Memory: Limited (<8GB) - May need to use smaller batch sizes")

# CPU check
if system_specs['cpu_count'] >= 8:
    print("✅ CPU: Excellent (8+ cores) - Great for parallel processing")
elif system_specs['cpu_count'] >= 4:
    print("✅ CPU: Good (4-8 cores) - Adequate for most tasks")
else:
    print("⚠️ CPU: Limited (<4 cores) - Training may be slower")

# GPU check
if gpu_info:
    print(f"✅ GPU: Available ({len(gpu_info)} GPU(s)) - Accelerated training enabled")
else:
    print("ℹ️ GPU: Not detected - CPU training available (slower but functional)")

print("\n💡 OPTIMIZATION TIPS")
print("=" * 50)

if not gpu_info:
    print("• Consider using Google Colab for GPU access during learning")
    print("• Start with smaller datasets and models for CPU training")
    print("• Use efficient algorithms and pre-trained models")

if total_memory_gb < 8:
    print("• Use smaller batch sizes (batch_size=16 or 32)")
    print("• Clear variables when not needed: del variable_name")
    print("• Consider using gradient accumulation for larger effective batch sizes")

print("• Close unnecessary applications during training")
print("• Monitor resource usage with this notebook")
print("• Use mixed precision training to save memory (torch.cuda.amp)")

print("\n✅ Hardware verification complete!")

## 4. Performance Baseline Test 📊

In [ ]:
# Performance baseline test - only run if packages are available
if package_status.get('torch', {}).get('installed', False) and package_status.get('numpy', {}).get('installed', False):
    import torch
    import numpy as np
    import time
    
    print("🏃‍♂️ PERFORMANCE BASELINE TEST")
    print("=" * 80)
    print("Running quick performance tests to establish baseline...")
    print()
    
    # Test 1: NumPy matrix multiplication
    print("1️⃣ NumPy Matrix Multiplication Test")
    print("-" * 40)
    
    size = 1000
    np.random.seed(42)
    a = np.random.randn(size, size).astype(np.float32)
    b = np.random.randn(size, size).astype(np.float32)
    
    start_time = time.time()
    c = np.dot(a, b)
    numpy_time = time.time() - start_time
    
    print(f"NumPy ({size}x{size} matrix multiplication): {numpy_time:.3f} seconds")
    
    # Test 2: PyTorch CPU performance
    print("\n2️⃣ PyTorch CPU Test")
    print("-" * 40)
    
    torch.manual_seed(42)
    a_torch = torch.randn(size, size, dtype=torch.float32)
    b_torch = torch.randn(size, size, dtype=torch.float32)
    
    start_time = time.time()
    c_torch = torch.mm(a_torch, b_torch)
    torch_cpu_time = time.time() - start_time
    
    print(f"PyTorch CPU ({size}x{size} matrix multiplication): {torch_cpu_time:.3f} seconds")
    
    # Test 3: PyTorch GPU performance (if available)
    if torch.cuda.is_available():
        print("\n3️⃣ PyTorch GPU Test")
        print("-" * 40)
        
        device = torch.device('cuda:0')
        a_gpu = a_torch.to(device)
        b_gpu = b_torch.to(device)
        
        # Warmup
        for _ in range(3):
            _ = torch.mm(a_gpu, b_gpu)
        torch.cuda.synchronize()
        
        start_time = time.time()
        c_gpu = torch.mm(a_gpu, b_gpu)
        torch.cuda.synchronize()
        gpu_time = time.time() - start_time
        
        print(f"PyTorch GPU ({size}x{size} matrix multiplication): {gpu_time:.3f} seconds")
        print(f"GPU Speedup: {torch_cpu_time/gpu_time:.1f}x faster than CPU")
        
        # Memory usage test
        print(f"\nGPU Memory Usage:")
        print(f"Allocated: {torch.cuda.memory_allocated()/1024**2:.1f} MB")
        print(f"Cached: {torch.cuda.memory_reserved()/1024**2:.1f} MB")
        
    else:
        print("\n3️⃣ GPU Test Skipped (no GPU available)")
    
    # Test 4: Simple neural network training speed
    print("\n4️⃣ Mini Neural Network Training Test")
    print("-" * 40)
    
    # Create a simple dataset
    torch.manual_seed(42)
    X = torch.randn(1000, 20)
    y = torch.randn(1000, 1)
    
    # Simple linear model
    model = torch.nn.Sequential(
        torch.nn.Linear(20, 50),
        torch.nn.ReLU(),
        torch.nn.Linear(50, 1)
    )
    
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    
    start_time = time.time()
    
    # Training loop
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
    
    training_time = time.time() - start_time
    
    print(f"100 epochs of mini neural network: {training_time:.3f} seconds")
    print(f"Final loss: {loss.item():.6f}")
    
    # Performance summary
    print("\n📊 PERFORMANCE SUMMARY")
    print("=" * 50)
    
    if torch.cuda.is_available():
        print(f"✅ GPU acceleration available and tested")
        print(f"💡 Recommended for training: Use GPU when possible")
    else:
        print(f"ℹ️ CPU-only performance tested")
        print(f"💡 Recommended: Consider cloud GPU for intensive training")
    
    if numpy_time < 2.0 and torch_cpu_time < 2.0:
        print(f"✅ Good computational performance detected")
    elif numpy_time < 5.0 and torch_cpu_time < 5.0:
        print(f"⚠️ Moderate performance - suitable for learning")
    else:
        print(f"⚠️ Slower performance detected - consider smaller models")
    
    print(f"\n💾 Performance baseline saved for future reference")
    
    # Save performance baseline
    baseline_results = {
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'numpy_time': numpy_time,
        'torch_cpu_time': torch_cpu_time,
        'training_time': training_time,
        'has_gpu': torch.cuda.is_available(),
        'system_specs': system_specs
    }
    
    if torch.cuda.is_available():
        baseline_results['torch_gpu_time'] = gpu_time
        baseline_results['gpu_speedup'] = torch_cpu_time/gpu_time
    
    # Store results for later use
    with open('performance_baseline.txt', 'w') as f:
        f.write("MLU Performance Baseline Results\n")
        f.write("=" * 40 + "\n")
        f.write(f"Test Date: {baseline_results['timestamp']}\n")
        f.write(f"NumPy Matrix Multiplication: {numpy_time:.3f}s\n")
        f.write(f"PyTorch CPU: {torch_cpu_time:.3f}s\n")
        if torch.cuda.is_available():
            f.write(f"PyTorch GPU: {gpu_time:.3f}s\n")
            f.write(f"GPU Speedup: {torch_cpu_time/gpu_time:.1f}x\n")
        f.write(f"Neural Network Training (100 epochs): {training_time:.3f}s\n")
        f.write(f"Memory: {system_specs['memory_total']}\n")
        f.write(f"CPU Cores: {system_specs['cpu_count']}\n")
    
    print(f"📄 Results saved to: performance_baseline.txt")

else:
    print("⚠️ Performance tests skipped - PyTorch and/or NumPy not installed")
    print("Install required packages first, then re-run this section")

## 5. Environment Setup Recommendations 🛠️

In [ ]:
print("🛠️ PERSONALIZED SETUP RECOMMENDATIONS")
print("=" * 80)

# Analyze all collected information to provide recommendations
recommendations = []
priority_level = "medium"

# Environment recommendations based on what we found
if env_details['conda']:
    if env_details['docker']:
        print("🎯 OPTIMAL SETUP DETECTED")
        print("You have both Anaconda and Docker available!")
        print()
        
        recommendations.extend([
            "✅ Use conda for day-to-day development and package management",
            "✅ Use Docker for isolated project environments and deployment",
            "✅ Consider conda for fast prototyping, Docker for production-like setups",
        ])
        priority_level = "low"
    else:
        print("🎯 CONDA-FOCUSED SETUP")
        print("Anaconda detected - excellent choice for data science!")
        print()
        
        recommendations.extend([
            "✅ Create a dedicated conda environment for MLU: conda create -n mlu python=3.9",
            "✅ Use conda-forge channel for latest packages: conda config --add channels conda-forge",
            "💡 Consider installing Docker for advanced containerization needs",
        ])
        priority_level = "low"

elif env_details['docker']:
    print("🎯 DOCKER-FOCUSED SETUP")
    print("Docker detected - great for reproducible environments!")
    print()
    
    recommendations.extend([
        "✅ Use the provided Docker setup for consistent environments",
        "💡 Consider installing Anaconda for easier package management",
        "✅ Use Jupyter Docker containers for isolated development",
    ])
    priority_level = "medium"

else:
    print("🎯 BASIC SETUP DETECTED")
    print("Setting up a comprehensive environment...")
    print()
    
    recommendations.extend([
        "📦 Install Anaconda for comprehensive data science setup",
        "🐳 Consider Docker for environment isolation",
        "⚡ Run the provided installation scripts to set up everything",
    ])
    priority_level = "high"

# Hardware-based recommendations
total_memory_gb = float(system_specs['memory_total'].split()[0])

print("\n💻 HARDWARE-OPTIMIZED RECOMMENDATIONS")
print("-" * 50)

if gpu_info:
    recommendations.extend([
        "🚀 GPU detected - use CUDA-enabled PyTorch for faster training",
        "⚡ Enable mixed precision training for memory efficiency",
        "🎮 Use larger batch sizes to fully utilize GPU memory",
    ])
    print("GPU available - Accelerated training recommended!")
else:
    recommendations.extend([
        "🧠 CPU-only setup - focus on efficient algorithms and smaller models",
        "☁️ Consider Google Colab for GPU access during intensive training",
        "📊 Use smaller batch sizes and simpler models for faster iteration",
    ])
    print("CPU-only setup - Optimized workflows recommended!")

if total_memory_gb >= 16:
    recommendations.append("💾 Excellent memory (16GB+) - can handle large datasets and models")
elif total_memory_gb >= 8:
    recommendations.append("💾 Good memory (8-16GB) - suitable for most learning tasks")
else:
    recommendations.extend([
        "💾 Limited memory (<8GB) - use smaller batch sizes",
        "🔄 Clear variables frequently and use data generators",
    ])

# Package-based recommendations
print("\n📦 PACKAGE SETUP RECOMMENDATIONS")
print("-" * 50)

all_packages_ok = all(status.get('installed', False) for status in package_status.values())

if all_packages_ok:
    print("✅ All packages are ready - you're good to go!")
    recommendations.append("🎉 Environment fully ready - start with Week 1 materials!")
else:
    missing = [name for name, status in package_status.items() if not status.get('installed', False)]
    print(f"📋 Missing packages detected: {', '.join(missing)}")
    
    if env_details['conda']:
        recommendations.append("🔧 Run: conda install pytorch torchvision d2l numpy pandas matplotlib jupyter")
    else:
        recommendations.append("🔧 Run: pip install torch torchvision d2l numpy pandas matplotlib jupyter")

# Priority-based action plan
print(f"\n📋 ACTION PLAN (Priority: {priority_level.upper()})")
print("=" * 50)

action_plan = {
    "high": [
        "1️⃣ Install Anaconda or Python package manager",
        "2️⃣ Run the MLU installation script (install.sh or install.ps1)",
        "3️⃣ Verify installation by re-running this notebook",
        "4️⃣ Start with Week 1 basic exercises",
    ],
    "medium": [
        "1️⃣ Install missing packages using your preferred method",
        "2️⃣ Run performance baseline test to verify setup",
        "3️⃣ Review GPU optimization tips if applicable",
        "4️⃣ Begin with Week 1 comprehensive materials",
    ],
    "low": [
        "1️⃣ Verify all packages are up to date",
        "2️⃣ Run a quick performance test",
        "3️⃣ Jump directly into Week 1 advanced topics",
        "4️⃣ Consider setting up alternative environments for experimentation",
    ]
}

for step in action_plan[priority_level]:
    print(step)

print("\n💡 DETAILED RECOMMENDATIONS")
print("=" * 50)

for i, rec in enumerate(recommendations, 1):
    print(f"{i:2d}. {rec}")

# Save recommendations to file
print(f"\n📄 SAVING PERSONALIZED RECOMMENDATIONS")
print("-" * 50)

with open('environment_recommendations.md', 'w') as f:
    f.write("# MLU Environment Setup Recommendations\n\n")
    f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("## Environment Summary\n")
    f.write(f"- **Conda Available**: {'✅' if env_details['conda'] else '❌'}\n")
    f.write(f"- **Docker Available**: {'✅' if env_details['docker'] else '❌'}\n")
    f.write(f"- **GPU Available**: {'✅' if gpu_info else '❌'}\n")
    f.write(f"- **Memory**: {system_specs['memory_total']}\n")
    f.write(f"- **CPU Cores**: {system_specs['cpu_count']}\n\n")
    
    f.write(f"## Action Plan (Priority: {priority_level.title()})\n")
    for i, step in enumerate(action_plan[priority_level], 1):
        f.write(f"{i}. {step.replace('1️⃣', '').replace('2️⃣', '').replace('3️⃣', '').replace('4️⃣', '').strip()}\n")
    f.write("\n")
    
    f.write("## Detailed Recommendations\n")
    for i, rec in enumerate(recommendations, 1):
        clean_rec = rec.replace('✅', '').replace('💡', '').replace('🚀', '').replace('⚡', '').replace('🎮', '').replace('🧠', '').replace('☁️', '').replace('📊', '').replace('💾', '').replace('🔄', '').replace('🎉', '').replace('📋', '').replace('🔧', '').strip()
        f.write(f"{i}. {clean_rec}\n")
    
    f.write("\n## Next Steps\n")
    f.write("1. Follow the action plan above\n")
    f.write("2. Run the appropriate installation script from the MLU repository\n")
    f.write("3. Verify installation by re-running this compatibility check\n")
    f.write("4. Start with Week 1 materials in the learning curriculum\n")

print("✅ Recommendations saved to: environment_recommendations.md")
print("📚 You can reference this file anytime during your learning journey!")

print(f"\n🎯 READY TO START?")
print("=" * 50)
if priority_level == "low":
    print("🎉 Your environment looks great! Start learning immediately!")
    print("📖 Recommended next step: Open week1_deep_learning_mastery.ipynb")
elif priority_level == "medium":
    print("⚡ Almost ready! Address the recommendations above first.")
    print("🔧 Estimated setup time: 10-30 minutes")
else:
    print("🛠️ Environment needs setup. Follow the action plan above.")
    print("⏱️ Estimated setup time: 30-60 minutes")

print("\n✅ Environment compatibility check complete!")
print("📊 All results saved for future reference.")